# Andromeda Spatial OS - Cloud Muscle Node
Run this notebook to start the vLLM + Moondream2 + Whisper pipeline on a free Colab GPU. Copy the generated Cloudflare URL into your local Andromeda UI settings.

In [ ]:
# GPU Sanity Check
import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU not found! Please change your runtime to GPU before running the notebook:\n"
        "-> Go to Menu: Runtime -> Change runtime type\n"
        "-> Select T4 GPU (or any available GPU)\n"
        "-> Click Save and re-run this cell."
    )
print("GPU is active! Proceeding...")

In [ ]:
# Smart Installer: Installs Ollama, faster-whisper, and necessary libraries.
import sys
import subprocess

def run_cmd(cmd):
    print(f"Running: {cmd}")
    subprocess.run(cmd, shell=True, check=True)

try:
    import ollama
    from faster_whisper import WhisperModel
    import fastapi
    print("✅ All packages (Ollama, Faster-Whisper, FastAPI) are correctly installed. Skipping installation!")
except Exception as e:
    print(f"🔄 Installing required packages ({e})...")
    
    # Remove any corrupt/failed binary from previous attempts
    run_cmd("rm -f /usr/local/bin/ollama")
    
    # 1. Install zstd (required to extract the .tar.zst archive on Colab containers)
    run_cmd("apt-get update -qq && apt-get install -y -qq zstd")
    
    # 2. Download and extract Ollama binary and GPU libraries
    run_cmd("curl -fsSL https://ollama.com/download/ollama-linux-amd64.tar.zst | tar --zstd -x -C /usr")
    
    # 3. Install Python packages
    run_cmd("pip install fastapi uvicorn nest-asyncio faster-whisper ollama")
    
    # 4. Download Cloudflare tunnel binary
    run_cmd("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared")
    run_cmd("chmod +x cloudflared")
    
    print("\n✅ Installation finished successfully!")
    print("🔄 Restarting the Colab runtime to apply changes...")
    
    # 5. Force kernel restart to clear memory
    import os
    os.kill(os.getpid(), 9)

In [ ]:
import os
import io
import base64
import json
import asyncio
import subprocess
import threading
import time
from fastapi import FastAPI, Request
from fastapi.middleware.cors import CORSMiddleware
from PIL import Image
import uvicorn
from faster_whisper import WhisperModel
import ollama

app = FastAPI()
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Start/Ensure Ollama service is running
print("Starting Ollama service in the background...")
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)

print("Pulling Qwen2.5-Coder and Moondream via Ollama...")
# Pull models if they don't exist (Ollama handles caching automatically)
subprocess.run(["ollama", "pull", "qwen2.5-coder:7b"], check=True)
subprocess.run(["ollama", "pull", "moondream"], check=True)

print("Loading Whisper...")
whisper_model = WhisperModel("tiny.en", device="cuda", compute_type="float16")

@app.post("/v1/chat/completions")
async def chat_completions(request: Request):
    data = await request.json()
    prompt = data.get("prompt", "")
    temperature = data.get("temperature", 0.7)
    max_tokens = data.get("max_tokens", 512)
    
    # Run Ollama inference in a background thread to prevent blocking the event loop
    def run_ollama_generate():
        response = ollama.generate(
            model="qwen2.5-coder:7b",
            prompt=prompt,
            options={
                "temperature": temperature,
                "num_predict": max_tokens
            }
        )
        return response.get("response", "")
        
    content = await asyncio.to_thread(run_ollama_generate)
    return {"choices": [{"message": {"content": content}}]}

@app.post("/analyze")
async def analyze_image(request: Request):
    data = await request.json()
    base64_img = data.get("image")
    query = data.get("query", "Extract all text from this image accurately.")
    
    if not base64_img:
        return {"response": "No image provided"}
        
    image_data = base64.b64decode(base64_img)
    
    def run_ollama_vision():
        response = ollama.chat(
            model="moondream",
            messages=[{
                "role": "user",
                "content": query,
                "images": [image_data]
            }]
        )
        return response.get("message", {}).get("content", "")
        
    result = await asyncio.to_thread(run_ollama_vision)
    return {"response": result}

@app.post("/transcribe")
async def transcribe_audio(request: Request):
    data = await request.json()
    base64_audio = data.get("audio")
    
    if not base64_audio:
        return {"text": "No audio provided"}
        
    audio_data = base64.b64decode(base64_audio)
    tmp_path = "temp_" + str(os.urandom(8).hex()) + ".webm"
    with open(tmp_path, "wb") as f:
        f.write(audio_data)
        
    def run_whisper():
        try:
            segments, info = whisper_model.transcribe(tmp_path, beam_size=5)
            return " ".join([segment.text for segment in segments])
        finally:
            import os
            if os.path.exists(tmp_path):
                os.remove(tmp_path)
                
    text = await asyncio.to_thread(run_whisper)
    return {"text": text}

def start_cloudflared():
    print("Starting Cloudflared Tunnel...")
    process = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:8000'], 
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        if "https://" in line and "trycloudflare.com" in line:
            words = line.split()
            found = False
            for word in words:
                if "trycloudflare.com" in word and "https://" in word:
                    # Clean up any surrounding characters like table borders (|)
                    url = word.strip(" |\"'")
                    print("\n" + "="*60)
                    print(f"YOUR ANDROMEDA CLOUD URL: {url}")
                    print("="*60 + "\n")
                    found = True
                    break
            if found:
                break

threading.Thread(target=start_cloudflared, daemon=True).start()

# Run Uvicorn directly on the running Jupyter event loop
config = uvicorn.Config(app=app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)
await server.serve()


## 🛠️ Troubleshooting & Diagnostics
If you encounter any `ImportError` or `RuntimeError` regarding `torchvision::nms` or mismatched `numpy` versions, run the diagnostics cell below. It will check package compatibility and print helpful fixing commands.

In [ ]:
# Diagnostics Cell
import sys
import torch
import torchvision
import numpy as np

print("="*50)
print(f"Python Version: {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
print(f"Torchvision Version: {torchvision.__version__}")
print(f"NumPy Version: {np.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA Device Name: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version (Torch): {torch.version.cuda}")
print("="*50)

# Verify torchvision works
try:
    from torchvision.ops import nms
    print("✅ Torchvision NMS operator imported successfully!")
except Exception as e:
    print(f"❌ Torchvision check failed: {e}")
    print("👉 QUICK FIX: Run: !pip install --force-reinstall torchvision --extra-index-url https://download.pytorch.org/whl/cu121")